<a href="https://colab.research.google.com/github/chanceCoderByPassion/DL_CV/blob/master/Yolo_safety_alert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.7 MB/s eta 0:00:00


In [ ]:

import cv2
import numpy as np
from ultralytics import YOLO

# 1. Setup Model and Video Source
model = YOLO("yolo11n.pt")
input_path = "/content/drive/MyDrive/Traffic.mp4"
output_path = "/content/drive/MyDrive/collision_output.mp4"

cap = cv2.VideoCapture(input_path)

# 2. Get Video Properties for the Writer
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define the codec and create VideoWriter object
# 'mp4v' is widely compatible for .mp4 files
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# Variables for tracking logic
track_history = {}
LOOK_AHEAD_FRAMES = fps  # Predict 1 second into the future
COLLISION_THRESHOLD = 60 # Pixel distance for warning

print(f"Processing video: {input_path}...")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break


    # 3. Object Tracking
    results = model.track(frame, persist=True, classes=[0, 2, 7], verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        predictions = {}

        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box

            # Update history
            if track_id not in track_history:
                track_history[track_id] = []
            track_history[track_id].append((x, y))

            if len(track_history[track_id]) > 10:
                track_history[track_id].pop(0)

            # 4. Trajectory Prediction
            if len(track_history[track_id]) >= 2:
                # Calculate movement vector
                start_pt = track_history[track_id][0]
                end_pt = track_history[track_id][-1]

                vx = (end_pt[0] - start_pt[0]) / len(track_history[track_id])
                vy = (end_pt[1] - start_pt[1]) / len(track_history[track_id])

                # Predict point
                future_x = x + (vx * LOOK_AHEAD_FRAMES)
                future_y = y + (vy * LOOK_AHEAD_FRAMES)
                predictions[track_id] = (future_x, future_y)

                # Draw trajectory on frame
                cv2.line(frame, (int(x), int(y)), (int(future_x), int(future_y)), (0, 255, 0), 2)

        # 5. Collision Analysis
        ids = list(predictions.keys())
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                p1, p2 = predictions[ids[i]], predictions[ids[j]]
                distance = np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

                if distance < COLLISION_THRESHOLD:
                    cv2.rectangle(frame, (0, 0), (frame_width, 60), (0, 0, 255), -1)
                    cv2.putText(frame, "CRITICAL COLLISION RISK", (int(frame_width/4), 40),
                                cv2.FONT_HERSHEY_DUPLEX, 1, (255, 255, 255), 2)

    # 6. Write frame to file
    out.write(frame)

    # Optional: Preview while processing
    #cv2.imshow("Processing...", frame)
    #if cv2.waitKey(1) & 0xFF == ord("q"):
        #break

# 7. Cleanup
#print(f"Finished! Saved to {output_path}")
cap.release()
out.release()
#cv2.destroyAllWindows()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Processing video: /content/drive/MyDrive/Traffic.mp4...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 175ms
Prepared 1 package in 39ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

